# 04 — Export Model to ONNX
Export the merged LoRA + base model to ONNX for faster CPU inference in production.

In [ ]:
!pip install optimum onnxruntime transformers peft torch --quiet

In [ ]:
import time
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL  = 'google/flan-t5-base'
MODEL_DIR   = Path('../backend/models')
ADAPTER_DIR = MODEL_DIR / 'lora_adapter'
ONNX_DIR    = MODEL_DIR / 'onnx'
ONNX_DIR.mkdir(parents=True, exist_ok=True)

## 1. Merge LoRA Weights into Base Model

In [ ]:
print('Loading base model...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
tokenizer  = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))

print('Loading LoRA adapter...')
peft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

print('Merging adapter weights into base model...')
merged_model = peft_model.merge_and_unload()
merged_model.eval()
print('Merge complete.')

merged_path = MODEL_DIR / 'merged_model'
merged_model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))
print(f'Merged model saved to {merged_path}')

## 2. Export to ONNX using Optimum

In [ ]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM

print('Exporting to ONNX...')
ort_model = ORTModelForSeq2SeqLM.from_pretrained(
    str(merged_path),
    export=True,
)
ort_model.save_pretrained(str(ONNX_DIR))
tokenizer.save_pretrained(str(ONNX_DIR))
print(f'ONNX model saved to {ONNX_DIR}')

for f in sorted(ONNX_DIR.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  {f.relative_to(ONNX_DIR)}: {size_mb:.1f} MB')

## 3. Benchmark: PyTorch vs ONNX Inference Speed

In [ ]:
test_prompt = 'Answer the following US immigration question accurately and clearly:\n\nQuestion: How do I renew my green card?\n\nAnswer:'
N = 5  # number of inference runs to average

# PyTorch inference
pt_inputs = tokenizer(test_prompt, return_tensors='pt', max_length=256, truncation=True)
pt_times = []
for _ in range(N):
    t0 = time.time()
    with torch.no_grad():
        out = merged_model.generate(**pt_inputs, max_new_tokens=200, num_beams=2)
    pt_times.append(time.time() - t0)

pt_answer = tokenizer.decode(out[0], skip_special_tokens=True)
print(f'PyTorch avg latency: {np.mean(pt_times)*1000:.0f} ms')
print(f'Answer: {pt_answer[:200]}...')
print()

In [ ]:
# ONNX Runtime inference
ort_tokenizer = AutoTokenizer.from_pretrained(str(ONNX_DIR))
ort_inputs = ort_tokenizer(test_prompt, return_tensors='pt', max_length=256, truncation=True)
ort_times = []
for _ in range(N):
    t0 = time.time()
    out = ort_model.generate(**ort_inputs, max_new_tokens=200, num_beams=2)
    ort_times.append(time.time() - t0)

ort_answer = ort_tokenizer.decode(out[0], skip_special_tokens=True)
print(f'ONNX avg latency: {np.mean(ort_times)*1000:.0f} ms')
print(f'Answer: {ort_answer[:200]}...')
print(f'\nSpeedup: {np.mean(pt_times)/np.mean(ort_times):.2f}x')

## 4. Optional: Quantize ONNX Model (INT8)

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

QUANT_DIR = MODEL_DIR / 'onnx_quantized'
QUANT_DIR.mkdir(exist_ok=True)

quantizer = ORTQuantizer.from_pretrained(str(ONNX_DIR))
qconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=False)

try:
    quantizer.quantize(save_dir=str(QUANT_DIR), quantization_config=qconfig)
    print(f'Quantized model saved to {QUANT_DIR}')
    # Compare sizes
    orig_size = sum(f.stat().st_size for f in ONNX_DIR.rglob('*.onnx')) / (1024**2)
    quant_size = sum(f.stat().st_size for f in QUANT_DIR.rglob('*.onnx')) / (1024**2)
    print(f'Original ONNX: {orig_size:.1f} MB')
    print(f'Quantized:     {quant_size:.1f} MB')
    print(f'Compression:   {orig_size/quant_size:.2f}x')
except Exception as e:
    print(f'Quantization not available on this platform: {e}')
    print('Use the unquantized ONNX model instead.')

## Summary
- Merged LoRA adapter into base model weights
- Exported merged model to ONNX via Hugging Face Optimum
- Benchmarked PyTorch vs ONNX latency
- Attempted INT8 quantization for further compression
- ONNX model at `backend/models/onnx/` is used by `backend/ml/inference.py`

**Next steps:** Deploy the FastAPI backend (`docker-compose up`) and connect the React Native app.